# Warranty V4 — semantic candidate inspection

This notebook complements (does not overwrite) `warranty_araz.ipynb`.

**Objective:** Improve **recall** while avoiding unsupported claims that all new candidate reviews truly discuss warranties. Contextual rules and a weakly supervised retrieval model are **retrieval tools**, not a validated classifier. The dataset must be in `train subset.zip` for source reproduction, but reviewing the included output CSVs does not require it.

In [ ]:
from pathlib import Path
import csv, json

cwd = Path.cwd().resolve()
root = next(p for p in [cwd, *cwd.parents] if (p/'outputs_warranty'/'warranty_v4_summary.json').exists())
output_dir = root/'outputs_warranty'
summary = json.loads((output_dir/'warranty_v4_summary.json').read_text(encoding='utf-8'))
print('Dataset:', summary['dataset_rows'])
print('V2 core:', summary['core_rule_matches'])
print('Contextual rule candidates:', summary['new_unverified_candidates'])
print('Weak-model-only candidates:', summary.get('weak_model_only_unverified_candidates',125))
print('All-retrieval exploratory union:', summary.get('expanded_exploratory_union_with_model',1725))
print('VERIFIED TRUE COUNT: UNKNOWN — requires human labelling')

## Semantic candidate ranking with Sentence-BERT

Uses a pretrained model already explored in the original notebook (`all-MiniLM-L6-v2`). **Positive minus negative query similarity is a ranking heuristic, not a probability**, and no threshold is automatically accepted. Reviews about software compatibility, Wi-Fi range and ordinary replacement shopping can otherwise look deceptively similar to warranty-related text.

In [ ]:
import csv
import re
import numpy as np
from sentence_transformers import SentenceTransformer

def read_csv(name):
    with (output_dir/name).open(encoding='utf-8-sig',newline='') as f:
        return list(csv.DictReader(f))

candidates = read_csv('warranty_v4_candidates.csv')
weak = read_csv('warranty_v4_weak_model_discoveries.csv')

def focused_window(text):
    markers = re.compile(r'\b(?:warran\w*|guarant\w*|rma|insurance|square\s*trade|protection|repair|replac\w*|customer\s+service|support|manufacturer|claim)\b',re.I)
    m = markers.search(text)
    return text[max(0,m.start()-220):m.start()+550] if m else text[:750]

items = []
for r in candidates:
    items.append({'review_id':r['review_id'],'source':'context_rules','rule':r['primary_rule'],'asin':r['asin'],'snippet':r['context'],'full_text':r['full_text']})
for r in weak:
    items.append({'review_id':r['review_id'],'source':'weak_retrieval','rule':'tfidf_logistic','asin':r['asin'],'snippet':focused_window(r['full_text']),'full_text':r['full_text']})
assert len(items)==len({r['review_id'] for r in items})

queries_positive = [
    'A customer discusses a product warranty, guarantee, or protection plan.',
    'A customer describes an RMA process after a product failure.',
    'A manufacturer provides a free repair or replacement for a defective product.',
    'The product has insurance coverage for breakdowns or defects.',
]
queries_negative = [
    'A customer discusses technical compatibility or software support.',
    'A customer discusses WiFi signal coverage or cellular reception.',
    'A customer purchases a replacement part or a different product.',
    'A customer makes a normal retailer return or discusses shipment.',
]

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
X = model.encode([r['snippet'] for r in items], batch_size=32, normalize_embeddings=True, show_progress_bar=True)
Qp = model.encode(queries_positive, normalize_embeddings=True)
Qn = model.encode(queries_negative, normalize_embeddings=True)
positive_sim = (X @ Qp.T).max(axis=1)
negative_sim = (X @ Qn.T).max(axis=1)
for r,p,n in zip(items,positive_sim,negative_sim):
    r['positive_similarity']=round(float(p),4)
    r['negative_similarity']=round(float(n),4)
    r['ranking_margin']=round(float(p-n),4)
    r['human_label']=''
    r['review_notes']=''
items.sort(key=lambda r:r['ranking_margin'],reverse=True)
ranked_path = output_dir/'warranty_v4_sbert_ranked.csv'
with ranked_path.open('w',encoding='utf-8-sig',newline='') as f:
    w=csv.DictWriter(f,fieldnames=list(items[0]))
    w.writeheader();w.writerows(items)
print('Ranked candidate reviews:',len(items))
print('Saved:',ranked_path)
for r in items[:12]:
    print(r['review_id'],r['rule'],r['ranking_margin'],r['snippet'][:210])

## Scientific stopping rule

Do not claim any exact number of true warranty-related reviews from the 1,027 core regex hits, the 1,600 contextual union, or the 1,725 total candidates. These are **retrieved** counts, not verified positives. Label the `warranty_v4_stratified_review.csv` sample independently, then execute `python src/nlp/warranty_v4_estimate.py` to estimate population prevalence, including missed positives outside all retrieval rules. Only promote a new class of candidates to final extraction if reviewed precision justifies it.